In [1]:
#!#pip install numpy
!pip install pandas matplotlib seaborn scikit-learn xgboost
!pip install tensorflow

   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
    --------------------------------------- 1.6/124.9 MB 9.4 MB/s eta 0:00:14
   - -------------------------------------- 3.1/124.9 MB 8.8 MB/s eta 0:00:14
   - -------------------------------------- 4.5/124.9 MB 7.9 MB/s eta 0:00:16
   -- ------------------------------------- 6.3/124.9 MB 7.9 MB/s eta 0:00:16
   -- ------------------------------------- 7.9/124.9 MB 8.0 MB/s eta 0:00:15
   --- ------------------------------------ 9.4/124.9 MB 8.0 MB/s eta 0:00:15
   --- ------------------------------------ 11.0/124.9 MB 7.9 MB/s eta 0:00:15
   ---- ----------------------------------- 12.6/124.9 MB 7.9 MB/s eta 0:00:15
   ---- ----------------------------------- 14.4/124.9 MB 8.0 MB/s eta 0:00:14
   ----- ---------------------------------- 16.0/124.9 MB 8.1 MB/s eta 0:00:14
   ----- ---------------------------------- 17.8/124.9 MB 8.1 MB/s eta 0:00:14
   ------ --------------------------------- 19.4/124.9 MB 8.1 MB/s

IMPORTING NECESSARY LIBRARY

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn import metrics
from sklearn.svm import SVC
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_absolute_error as mae

import warnings
warnings.filterwarnings('ignore')


dfSET LOADING

In [3]:
df = pd.read_csv('/home/infinity/Documents/IP/retail_store_inventory.csv')
display(df.head())
display(df.tail())


FileNotFoundError: [Errno 2] No such file or directory: '/home/infinity/Documents/IP/retail_store_inventory.csv'

SHAPE

In [ ]:
df.shape

(73100, 15)

DESCRIPTION

In [ ]:
df.describe()

,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Holiday/Promotion,Competitor Pricing
count,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000
mean,274.469877,136.464870,110.004473,141.494720,55.135108,10.009508,0.497305,55.146077
std,129.949514,108.919406,52.277448,109.254076,26.021945,7.083746,0.499996,26.191408
min,50.000000,0.000000,20.000000,-9.990000,10.000000,0.000000,0.000000,5.030000
25%,162.000000,49.000000,65.000000,53.670000,32.650000,5.000000,0.000000,32.680000
50%,273.000000,107.000000,110.000000,113.015000,55.050000,10.000000,0.000000,55.010000
75%,387.000000,203.000000,155.000000,208.052500,77.860000,15.000000,1.000000,77.820000
max,500.000000,499.000000,200.000000,518.550000,100.000000,20.000000,1.000000,104.940000


INFO

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  object 
 1   Store ID            73100 non-null  object 
 2   Product ID          73100 non-null  object 
 3   Category            73100 non-null  object 
 4   Region              73100 non-null  object 
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  object 
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  object 
dtypes: float64(3), int64(5), object(7)
memory usage: 8.4+

FEATURE ENGINEERING

DERIVED ATTRIBUTES


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df.head(1000)

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022,1,1
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022,1,1
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022,1,1
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022,1,1
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2022-01-10,S005,P0016,Groceries,East,407,103,90,107.56,60.15,15,Sunny,1,64.52,Winter,2022,1,10
996,2022-01-10,S005,P0017,Furniture,East,308,266,164,259.60,31.07,20,Rainy,1,30.57,Summer,2022,1,10
997,2022-01-10,S005,P0018,Furniture,South,331,235,49,234.25,22.58,20,Cloudy,1,27.56,Summer,2022,1,10
998,2022-01-10,S005,P0019,Electronics,East,481,154,169,169.61,85.04,5,Cloudy,0,82.62,Autumn,2022,1,10


In [ ]:
#Deleting the Date column
df.drop('Date', axis=1, inplace=True)
df.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day
0,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022,1,1
1,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022,1,1
2,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022,1,1
3,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022,1,1
4,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022,1,1


CYCLICAL ENCODING

In [ ]:
df['M1'] = np.sin(df['Month'] * (2 * np.pi / 12))
df['M2'] = np.cos(df['Month'] * (2 * np.pi / 12))
df.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,M1,M2
0,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022,1,1,0.5,0.866025
1,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022,1,1,0.5,0.866025
2,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022,1,1,0.5,0.866025
3,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022,1,1,0.5,0.866025
4,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022,1,1,0.5,0.866025


VALUES ENCODING

STOCKOUT TIME CALCULATION

In [ ]:
# If historical data is not available, use a user-defined forecast growth rate from external research or expert opinions.
# For instance, let's assume industry research suggests a 3% monthly growth rate (0.03)
forecast_growth = 0.03
def calculate_stockout_time(row, forecast_growth=0):
    """
    Calculate the estimated stockout time in days.
    
    The adjusted demand rate factors in forecast growth:
        adjusted_demand = Units Sold * (1 + forecast_growth)
    
    Stockout time is then:
        Stockout Time = Inventory Level / adjusted_demand
    """
    
    # Calculate adjusted demand rate
    adjusted_demand = row["Units Sold"] * (1 + forecast_growth)
    # Avoid division by zero
    if adjusted_demand == 0:
        adjusted_demand+=1
    return row["Inventory Level"] / adjusted_demand

# Apply the calculation across the DataFrame
df["Stockout Time (days)"] = df.apply(lambda row: calculate_stockout_time(row, forecast_growth), axis=1)


In [ ]:
df.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,M1,M2,Stockout Time (days)
0,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022,1,1,0.5,0.866025,1.765920
1,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022,1,1,0.5,0.866025,1.320388
2,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022,1,1,0.5,0.866025,1.523525
3,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022,1,1,0.5,0.866025,7.464587
4,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022,1,1,0.5,0.866025,11.511789


In [ ]:
data = df.copy()

LABEL ENCODING

In [ ]:
# List of categorical columns for label encoding:
categorical_columns = ["Store ID", "Product ID", "Category", "Region", 
                       "Weather Condition", "Seasonality"]

# Initialize a LabelEncoder
le = LabelEncoder()

# Apply label encoding to each categorical column:
for col in categorical_columns:
    df[col] = le.fit_transform(df[col])

df.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,M1,M2,Stockout Time (days)
0,0,0,3,1,231,127,55,135.47,33.50,20,1,0,29.69,0,2022,1,1,0.5,0.866025,1.765920
1,0,1,4,2,204,150,66,144.04,63.01,20,3,0,66.16,0,2022,1,1,0.5,0.866025,1.320388
2,0,2,4,3,102,65,51,74.02,27.99,10,3,1,31.32,2,2022,1,1,0.5,0.866025,1.523525
3,0,3,4,1,469,61,164,62.18,32.72,10,0,1,34.74,0,2022,1,1,0.5,0.866025,7.464587
4,0,4,1,0,166,14,135,9.26,73.64,0,3,0,68.95,2,2022,1,1,0.5,0.866025,11.511789


NORMALIZATION

In [ ]:
numeric_columns = ["Inventory Level", "Units Sold", "Units Ordered", "Demand Forecast", 
                   "Price", "Discount", "Competitor Pricing"]

# Initialize the StandardScaler for normalization (z-score normalization)
scaler = MinMaxScaler(feature_range=(0,1))

# Fit and transform the numeric columns
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])
df.head()


,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,M1,M2,Stockout Time (days)
0,0,0,3,1,0.402222,0.254509,0.194444,0.275211,0.261111,1.0,1,0,0.246822,0,2022,1,1,0.5,0.866025,1.765920
1,0,1,4,2,0.342222,0.300601,0.255556,0.291425,0.589000,1.0,3,0,0.611851,0,2022,1,1,0.5,0.866025,1.320388
2,0,2,4,3,0.115556,0.130261,0.172222,0.158947,0.199889,0.5,3,1,0.263137,2,2022,1,1,0.5,0.866025,1.523525
3,0,3,4,1,0.931111,0.122244,0.800000,0.136546,0.252444,0.5,0,1,0.297368,0,2022,1,1,0.5,0.866025,7.464587
4,0,4,1,0,0.257778,0.028056,0.638889,0.036421,0.707111,0.0,3,0,0.639776,2,2022,1,1,0.5,0.866025,11.511789


*MODEL TRAINING*

SPLITING TRAINING AND VALIDATION

In [ ]:
features = df.drop(['Stockout Time (days)'], axis=1)
target = df['Stockout Time (days)'].values


X_train, X_val, Y_train, Y_val = train_test_split(features, target,
                                                  test_size = 0.05,
                                                  random_state=22)
X_train.shape, X_val.shape


((69445, 19), (3655, 19))

MODEL USED LINEAR REGRESSION & XGB

In [ ]:
models = [LinearRegression(), XGBRegressor()]
pred =[]

for i in range(2):
    models[i].fit(X_train, Y_train)

    print(f'{models[i]} : ')

    train_preds = models[i].predict(X_train)
    print('Training Error : ', mae(Y_train, train_preds))

    val_preds = models[i].predict(X_val)
    pred.append(val_preds)
    print('Validation Error : ', mae(Y_val, val_preds))
    print()
    
print(max(pred[1]))


LinearRegression() : 
Training Error :  8.444769244617309
Validation Error :  8.765281052721404

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...) : 
Training Error :  0.0693521232419604
Validation Error :  0.10382393638212696

476.3888


In [ ]:

scaled_data =df.copy()

# Create time windows
def create_sequences(data, time_steps=180):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps, :-1])  # All features except target
        y.append(data[i+time_steps: -1])    # Target feature
    return np.array(X), np.array(y)

time_steps =180
X, y = create_sequences(scaled_data)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


InvalidIndexError: (slice(0, 180, None), slice(None, -1, None))

In [ ]:

# Create LSTM sequences
# Split into train and test
split = int(len(X) * 0.8)
X_train, X_test, y_train, y_test = X[:split], X[split:], y[:split], y[split:]
# Define LSTM model
lstm_model = Sequential([
    LSTM(50, return_sequences=False, input_shape=(time_steps, X.shape[2])),
    Dense(10, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Extract LSTM features for XGBoost
lstm_features = lstm_model.predict(X)
train_features = lstm_features[:split]
test_features = lstm_features[split:]

KeyError: (10, 0)